# Cycling DA and Multiscale Alignment with the QG Model

The quasi-geostrophic (QG) model, written in Fortran by [Dr. Shafer Smith](https://cims.nyu.edu/~shafer/tools/index.html), is a two-layer spectral model on a doubly periodic domain. It evolves the streamfunction $\\psi$ under geostrophic and hydrostatic balance, producing realistic mesoscale turbulence representative of mid-latitude atmospheric or oceanic dynamics.

**Key parameters:**
- Resolution: $k_{\\rm max}=127$ (256 × 256 physical grid)
- Two vertical layers (upper and lower)
- Time unit: 0.1 model time = 1 day
- State variable: streamfunction $\\psi$ at both layers

This notebook has three parts:

| Part | What | Why |
|------|------|-----|
| **I. Cycling DA** | Run 13 daily ETKF cycles and diagnose RMSE vs cycle | Baseline: standard ensemble DA converging on the QG state |
| **II. Assimilator comparison** | Rerun with EAKF and compare RMSE to ETKF | Verify batch and serial filters agree |
| **III. Multiscale alignment** | Rerun with the alignment updator (Horn-Schunck optical flow) | Demonstrate position-error correction beyond standard EnKF |

## System setup (optional, for Google Colab only)

In [ ]:
to_install = False

if to_install:
    %cd ~
    %rm -rf NEDAS
    !git clone https://github.com/nansencenter/NEDAS.git
    %cd NEDAS
    %pip install .
    %pip install numba cmocean ipywidgets
    %mkdir ~/work
    %cd ~/work
    %rm -rf NEDAS_tutorials
    !git clone https://github.com/myying/NEDAS_tutorials.git
    %cd ~/work/NEDAS_tutorials

## Load the NEDAS system and dependencies

In [ ]:
import os
import shutil
import numpy as np
import matplotlib.pyplot as plt
import cmocean
from datetime import datetime, timedelta, timezone
from IPython.display import display

from NEDAS.config import Config
from NEDAS import get_scheme

## Shared helper functions

These functions are reused across all three experiment parts.

In [ ]:
def get_analysis_times(config):
    """Return list of analysis cycle datetimes."""
    times = []
    t = config.time_analysis_start
    while t <= config.time_analysis_end:
        times.append(t)
        t += timedelta(hours=config.cycle_period)
    return times


def read_analysis_mean(work_dir, time, tag='post', varname='streamfunc', k=0):
    """Read the prior or posterior ensemble mean from NEDAS analysis binary files.

    Parameters
    ----------
    work_dir : str
    time     : datetime
    tag      : 'prior' or 'post'
    varname  : state variable name (default 'streamfunc')
    k        : vertical level index (0 = upper layer, 1 = lower layer)

    Returns
    -------
    2-D ndarray (ny, nx) in physical space, or None if the file is absent.
    """
    adir = os.path.join(work_dir, 'cycle', time.strftime('%Y%m%d%H%M'), 'analysis')
    dat_file = os.path.join(adir, f'fields_{tag}_mean.dat')
    bin_file = os.path.join(adir, f'fields_{tag}_mean.bin')

    if not os.path.exists(dat_file):
        return None

    with open(dat_file) as f:
        lines = [l.strip() for l in f if l.strip()]

    nx, ny = map(int, lines[0].split())

    for line in lines[2:]:
        parts = line.split()
        if parts[0] == varname and abs(float(parts[8]) - k) < 0.01:
            offset = int(parts[9])   # byte offset within the .bin file
            with open(bin_file, 'rb') as f:
                f.seek(offset)
                data = np.frombuffer(f.read(nx * ny * 4), dtype=np.float32)
            return data.reshape(ny, nx).copy()
    return None


def read_truth(model, work_dir, time, k=0):
    """Read the true streamfunction from QG Fortran binary files."""
    return model.read_var(
        path=os.path.join(work_dir, 'truth'),
        member=None, name='streamfunc', time=time, k=k
    )


def collect_rmse(model, work_dir, analysis_times):
    """Compute prior and posterior RMSE for each analysis cycle."""
    prior_rmse, post_rmse = [], []
    for t in analysis_times:
        truth      = read_truth(model, work_dir, t)
        prior_mean = read_analysis_mean(work_dir, t, 'prior')
        post_mean  = read_analysis_mean(work_dir, t, 'post')
        if prior_mean is not None:
            prior_rmse.append(np.sqrt(np.mean((truth - prior_mean) ** 2)))
            post_rmse.append( np.sqrt(np.mean((truth - post_mean)  ** 2)))
    return prior_rmse, post_rmse


def clear_cycle_dir(config):
    """Remove the cycle output directory so the next run starts fresh."""
    cycle_dir = os.path.join(config.work_dir, 'cycle')
    if os.path.exists(cycle_dir):
        shutil.rmtree(cycle_dir)


def plot_rmse(cycle_numbers, rmse_dict, title):
    """Plot one or more posterior RMSE time series."""
    styles = [('b-o', 2), ('r--s', 2), ('g:^', 2)]
    fig, ax = plt.subplots(figsize=(8, 4))
    for (label, values), (style, lw) in zip(rmse_dict.items(), styles):
        ax.plot(cycle_numbers, values, style, label=label, markersize=6, linewidth=lw)
    ax.set_xlabel('DA Cycle')
    ax.set_ylabel('Posterior RMSE')
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xticks(cycle_numbers)
    plt.tight_layout()
    plt.show()


def plot_fields(model, work_dir, analysis_times, cycle_idx, label):
    """Plot truth / prior mean / posterior mean at a given cycle index."""
    t = analysis_times[cycle_idx]
    truth      = read_truth(model, work_dir, t)
    prior_mean = read_analysis_mean(work_dir, t, 'prior')
    post_mean  = read_analysis_mean(work_dir, t, 'post')

    if prior_mean is None:
        print(f"No analysis output for {t.date()}.")
        return

    clim = np.percentile(np.abs(truth), 98)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    for ax, field, title in zip(axes,
                                [truth, prior_mean, post_mean],
                                ['Truth', 'Prior mean', 'Posterior mean']):
        im = ax.pcolormesh(field, vmin=-clim, vmax=clim, cmap='RdBu_r', shading='auto')
        ax.set_title(title, fontsize=12)
        ax.set_aspect('equal')
        ax.set_xticks([]); ax.set_yticks([])
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    prior_e = np.sqrt(np.mean((truth - prior_mean) ** 2))
    post_e  = np.sqrt(np.mean((truth - post_mean)  ** 2))
    fig.suptitle(
        f'Streamfunction (upper layer) — {t.strftime("%Y-%m-%d")} | {label}\n'
        f'prior RMSE = {prior_e:.4f},  posterior RMSE = {post_e:.4f}',
        fontsize=12, y=1.04
    )
    plt.tight_layout()
    plt.show()

---
## Part I: Cycling DA with ETKF

We run 13 daily analysis cycles (Jan 01–14) with the Ensemble Transform Kalman Filter (ETKF):
- **20 ensemble members**
- **3 000 synthetic observations** of streamfunction, error std = 0.2
- **Localization**: Gaspari-Cohn, radius = 100 grid points
- **Inflation**: posterior multiplicative, coefficient = 1.05

Pre-generated truth and initial ensemble files in `qg/work/truth/` and `qg/work/init_ens/` are detected automatically.

> **Runtime**: ~5–15 minutes depending on CPU speed.

In [ ]:
config = Config(config_file="qg/control.yml", quiet=True)

# Optionally adjust parameters before running
config.nens = 20
config.obs_def[0]['hroi'] = 100
config.inflation_def['coef'] = 1.05

# Use the full available truth window (Jan 02–14 = 13 cycles)
config.time_analysis_start = datetime(2001, 1, 2, tzinfo=timezone.utc)
config.time_analysis_end   = datetime(2001, 1, 14, tzinfo=timezone.utc)
config.run_diagnose = False   # skip netCDF output for speed

analysis_times = get_analysis_times(config)
work_dir = config.work_dir
ncycle = len(analysis_times)
print(f"Experiment: {config.time_analysis_start.date()} — {config.time_analysis_end.date()}")
print(f"Cycles: {ncycle}, nens: {config.nens}, hroi: {config.obs_def[0]['hroi']} grid pts")

In [ ]:
scheme_etkf = get_scheme(config)
clear_cycle_dir(config)
scheme_etkf()

In [ ]:
model = scheme_etkf.c.models['qg.fortran']
etkf_prior_rmse, etkf_post_rmse = collect_rmse(model, work_dir, analysis_times)
cycle_numbers = list(range(1, len(etkf_post_rmse) + 1))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(cycle_numbers, etkf_prior_rmse, 'b--o', label='Prior RMSE',     markersize=6, linewidth=1.5)
ax.plot(cycle_numbers, etkf_post_rmse,  'r-o',  label='Posterior RMSE', markersize=6, linewidth=2)
ax.set_xlabel('DA Cycle')
ax.set_ylabel('RMSE')
ax.set_title('Part I: ETKF — RMSE vs Cycle')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xticks(cycle_numbers)
plt.tight_layout()
plt.show()

print(f"Mean posterior RMSE: {np.mean(etkf_post_rmse):.4f}")

In [ ]:
# Streamfunction fields at cycle 1 and final cycle
plot_fields(model, work_dir, analysis_times, 0,  'Cycle 1 (first)')
plot_fields(model, work_dir, analysis_times, -1, f'Cycle {ncycle} (last)')

---
## Part II: Assimilator comparison — ETKF vs EAKF

NEDAS supports both **batch** (ETKF) and **serial** (EAKF) ensemble Kalman filters:

| Assimilator | Strategy |
|---|---|
| ETKF | Update all observations simultaneously via ensemble square-root |
| EAKF | Process each observation sequentially, updating ensemble in place |

Both should yield similar posterior RMSE under the same localization and inflation settings.

> **Runtime**: another ~5–15 minutes.

In [ ]:
config_eakf = Config(config_file="qg/control.yml", quiet=True)
config_eakf.assimilator_def['type'] = 'EAKF'

scheme_eakf_filt = get_scheme(config_eakf)
clear_cycle_dir(config_eakf)
scheme_eakf_filt()

In [ ]:
model_eakf = scheme_eakf_filt.c.models['qg.fortran']
_, eakf_post_rmse = collect_rmse(model_eakf, work_dir, analysis_times)

plot_rmse(
    cycle_numbers,
    {'ETKF posterior': etkf_post_rmse, 'EAKF posterior': eakf_post_rmse},
    'Part II: ETKF vs EAKF — Posterior RMSE'
)

print(f"Mean ETKF posterior RMSE: {np.mean(etkf_post_rmse):.4f}")
print(f"Mean EAKF posterior RMSE: {np.mean(eakf_post_rmse):.4f}")

---
## Part III: Multiscale alignment

Standard EnKF corrects **amplitude** errors but not **position** errors — if an eddy sits in the wrong location across ensemble members, the ensemble mean smears out rather than shifting. The **alignment updator** addresses this by computing an optical-flow displacement field between each member's prior and posterior streamfunction and applying that displacement before the next forecast. See [Ying (2019)](https://doi.org/10.1175/MWR-D-18-0376.1) for the method.

Here we compare **ETKF** (Part I baseline) against **ETKF + Horn-Schunck alignment**.

> **Runtime**: another ~5–15 minutes.

In [ ]:
config_align = Config(config_file="qg/control.yml", quiet=True)

config_align.updator_def = {
    'type': 'alignment',
    'variable': 'streamfunc',
    'interp_displaced_fields': True,
    'optical_flow': {
        'method': 'HornSchunck_pyramid',
        'nlevel': 3,
        'niter_max': 100,
        'smoothness_weight': 1.0,
    },
}

scheme_align = get_scheme(config_align)
clear_cycle_dir(config_align)
scheme_align()

In [ ]:
model_align = scheme_align.c.models['qg.fortran']
_, align_post_rmse = collect_rmse(model_align, work_dir, analysis_times)

plot_rmse(
    cycle_numbers,
    {'ETKF posterior': etkf_post_rmse, 'ETKF + alignment posterior': align_post_rmse},
    'Part III: Standard ETKF vs ETKF + Alignment — Posterior RMSE'
)

print(f"Mean ETKF posterior RMSE:            {np.mean(etkf_post_rmse):.4f}")
print(f"Mean ETKF + alignment posterior RMSE:{np.mean(align_post_rmse):.4f}")

In [ ]:
# Last-cycle fields from the alignment run
plot_fields(model_align, work_dir, analysis_times, -1, f'Cycle {ncycle} — ETKF + Alignment')